<a href="https://www.kaggle.com/code/jayhawk1900/f1-knn-diversity?scriptVersionId=321493058" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# F1 Pit Stop Prediction — KNN as a Diversity Model

**Competition:** [Playground Series S6E5](https://www.kaggle.com/competitions/playground-series-s6e5) | **Metric:** ROC-AUC

---

## Motivation

Pit-stop decisions are fundamentally a "find similar situations" problem: a given driver, on a given track, at a given tire age and track position, tends to pit when comparable past situations pitted. That logic maps directly onto **k-Nearest Neighbors** — an instance-based model that predicts by averaging the labels of the most similar rows.

This matters for the ensemble because KNN is a **completely different learning paradigm** from the models already in the blend:
- **Gradient-boosted trees** (XGBoost) learn threshold splits
- **Neural networks** (RealMLP, TabM) learn smooth nonlinear functions via backprop
- **KNN** trains no weights at all — it reasons purely by similarity

A prior experiment showed that adding a second neural architecture (TabM) didn't help, because two neural nets make correlated errors (~0.97). The open question: does a genuinely different *paradigm* like KNN bring uncorrelated errors that improve the blend?

---

## Method

**Domain-aligned distance metric.** KNN is only as good as its notion of "similar," so the features are engineered to match the decision being modeled:
- **Numeric race-situation features** — tire age, position, lap number, lap time, degradation, race progress — standardized so each contributes fairly to distance.
- **Target-encoded categoricals** — Driver, Track, and Compound are replaced by their historical pit rate (Bayesian-smoothed). This turns "similar driver" into "driver with similar pit tendency," making categorical similarity meaningful in a numeric distance space and avoiding the curse of dimensionality from one-hot encoding 887 drivers.

**Setup.** 5-fold stratified CV with the original F1 strategy dataset concatenated into each training fold. Target encoding is fit only on each fold's training portion (no leakage). Multiple neighborhood sizes (k = 100, 300, 700) are trained and averaged.

**Diversity test.** KNN's out-of-fold predictions are correlated against RealMLP, XGBoost, and TabM. Lower correlation means fresher signal. A 4-way rank-normalized, coordinate-ascent blend then reveals whether KNN earns meaningful weight.

---

## What Success Looks Like

KNN will likely be **weaker solo** than the trained models (instance-bas

In [1]:
import os, time, warnings
import numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings('ignore')

# ---------- Pre-computed model predictions (paths corrected; TabM dropped) ----------
RMLP = '/kaggle/input/notebooks/jayhawk1900/f1-tabm'          # RealMLP files live here
XGBD = '/kaggle/input/notebooks/jayhawk1900/f1-final-blend'   # XGB files live here
for f in [f'{RMLP}/oof_realmlp_ms5.npy', f'{XGBD}/oof_xgb_orig.npy']:
    assert os.path.exists(f), f'MISSING: {f}'
print('Model prediction files confirmed (RealMLP + XGB).')

# ---------- Load data ----------
DATA_DIR = '/kaggle/input/competitions/playground-series-s6e5'
ORIG_DIR = '/kaggle/input/datasets/aadigupta1601/f1-strategy-dataset-pit-stop-prediction'
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
orig = pd.read_csv(f'{ORIG_DIR}/f1_strategy_dataset_v4.csv').drop(columns=['Normalized_TyreLife'])
y = train['PitNextLap'].astype(np.int8)
y_orig = orig['PitNextLap'].astype(np.int8)
print('Loaded:', train.shape, test.shape, orig.shape)

# ---------- Domain-aligned KNN features ----------
SEED = 42
def smoothed_te(tr_df, tr_y, key, val_df, smoothing=20):
    gm = tr_y.mean()
    s = pd.DataFrame({key: tr_df[key].values, 'y': tr_y.values}).groupby(key)['y'].agg(['mean','count'])
    s['te'] = (s['count']*s['mean'] + smoothing*gm)/(s['count']+smoothing)
    return val_df[key].map(s['te']).fillna(gm).values

num_feats = ['LapNumber','Stint','TyreLife','Position','LapTime (s)',
             'LapTime_Delta','Cumulative_Degradation','RaceProgress','Position_Change']
te_keys = ['Driver','Compound','Race']

def build_knn_features(src_df, src_y, target_df):
    parts = [target_df[num_feats].values.astype(np.float32)]
    for k in te_keys:
        parts.append(smoothed_te(src_df, src_y, k, target_df).reshape(-1,1))
    return np.hstack(parts)

# ---------- KNN: 5-fold OOF (ORIG concatenated into training) ----------
N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
orig_folds = list(StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED).split(orig, y_orig))

K_VALUES = [100, 300, 700]
knn_oof = {k: np.zeros(len(train), dtype=np.float32) for k in K_VALUES}
knn_pred = {k: np.zeros(len(test), dtype=np.float32) for k in K_VALUES}

print('Training KNN (domain-aligned features, multiple k)...')
t0 = time.time()
for fold, ((tr, va), (otr, _)) in enumerate(zip(skf.split(train, y), orig_folds), 1):
    tr_df = pd.concat([train.iloc[tr], orig.iloc[otr]], axis=0).reset_index(drop=True)
    tr_y = pd.concat([y.iloc[tr], y_orig.iloc[otr]], axis=0).reset_index(drop=True)
    Xtr = build_knn_features(tr_df, tr_y, tr_df)
    Xva = build_knn_features(tr_df, tr_y, train.iloc[va])
    Xte = build_knn_features(tr_df, tr_y, test)
    sc = StandardScaler().fit(Xtr)
    Xtr, Xva, Xte = sc.transform(Xtr), sc.transform(Xva), sc.transform(Xte)
    for k in K_VALUES:
        knn = KNeighborsClassifier(n_neighbors=k, weights='distance', n_jobs=-1)
        knn.fit(Xtr, tr_y)
        knn_oof[k][va] = knn.predict_proba(Xva)[:, 1]
        knn_pred[k] += knn.predict_proba(Xte)[:, 1] / N_SPLITS
    print(f'  Fold {fold} done ({(time.time()-t0)/60:.1f} min)')

print('\nKNN OOF AUCs by k:')
for k in K_VALUES:
    print(f'  k={k}: {roc_auc_score(y, knn_oof[k]):.5f}')
oof_knn = np.mean([knn_oof[k] for k in K_VALUES], axis=0)
pred_knn = np.mean([knn_pred[k] for k in K_VALUES], axis=0)
print(f'  KNN (k-averaged): {roc_auc_score(y, oof_knn):.5f}')

# ---------- Load RealMLP and XGB ----------
oof_rmlp = np.load(f'{RMLP}/oof_realmlp_ms5.npy'); pred_rmlp = np.load(f'{RMLP}/pred_realmlp_ms5.npy')
oof_xgb = np.load(f'{XGBD}/oof_xgb_orig.npy'); pred_xgb = np.load(f'{XGBD}/pred_xgb_orig.npy')

# ---------- Correlations (the key diversity check) ----------
def rn(x): return rankdata(x)/len(x)
R = {'RealMLP': rn(oof_rmlp), 'XGB': rn(oof_xgb), 'KNN': rn(oof_knn)}
print('\nIndividual OOF AUCs:')
for n, v in [('RealMLP',oof_rmlp),('XGB',oof_xgb),('KNN',oof_knn)]:
    print(f'  {n:8s}: {roc_auc_score(y, v):.5f}')
print('\nCorrelation of KNN with the others (LOWER = more diverse):')
for n in ['RealMLP','XGB']:
    print(f'  KNN-{n}: {np.corrcoef(R["KNN"], R[n])[0,1]:.4f}')
print(f'  (reference) RealMLP-XGB: {np.corrcoef(R["RealMLP"], R["XGB"])[0,1]:.4f}')

# ---------- 3-way coordinate-ascent blend ----------
names = ['RealMLP','XGB','KNN']
oofs = [R[n] for n in names]
def ba(w):
    w=np.array(w); w=w/w.sum(); return roc_auc_score(y, sum(wi*o for wi,o in zip(w,oofs)))
wts = np.ones(3)/3; best = ba(wts)
for _ in range(300):
    imp=False
    for i in range(3):
        for d in [0.05,-0.05,0.02,-0.02,0.01,-0.01]:
            tw=wts.copy(); tw[i]=max(0,tw[i]+d); a=ba(tw)
            if a>best+1e-7: best=a; wts=tw; imp=True
    if not imp: break
wts=wts/wts.sum()
print('\n3-way blend weights:')
for n,w in zip(names,wts): print(f'  {n:8s}: {w:.3f}')
print(f'\n3-way blend OOF: {best:.5f}')
print(f'(RealMLP+XGB 2-way was ~0.95470 | best LB: 0.95390)')

# ---------- Submission ----------
preds = {'RealMLP': rn(pred_rmlp), 'XGB': rn(pred_xgb), 'KNN': rn(pred_knn)}
pb = sum(w*preds[n] for w,n in zip(wts,names))
sub = pd.DataFrame({'id': test['id'], 'PitNextLap': pb})
sub.to_csv('/kaggle/working/submission.csv', index=False)
np.save('/kaggle/working/oof_knn.npy', oof_knn)
np.save('/kaggle/working/pred_knn.npy', pred_knn)
print(f'\nSaved submission.csv + oof_knn.npy/pred_knn.npy')

Model prediction files confirmed (RealMLP + XGB).
Loaded: (439140, 16) (188165, 15) (101371, 15)
Training KNN (domain-aligned features, multiple k)...
  Fold 1 done (53.0 min)
  Fold 2 done (95.5 min)
  Fold 3 done (149.7 min)
  Fold 4 done (206.0 min)
  Fold 5 done (260.0 min)

KNN OOF AUCs by k:
  k=100: 0.91068
  k=300: 0.90272
  k=700: 0.89301
  KNN (k-averaged): 0.90462

Individual OOF AUCs:
  RealMLP : 0.95409
  XGB     : 0.95125
  KNN     : 0.90462

Correlation of KNN with the others (LOWER = more diverse):
  KNN-RealMLP: 0.7524
  KNN-XGB: 0.7784
  (reference) RealMLP-XGB: 0.9613

3-way blend weights:
  RealMLP : 0.726
  XGB     : 0.274
  KNN     : 0.000

3-way blend OOF: 0.95456
(RealMLP+XGB 2-way was ~0.95470 | best LB: 0.95390)

Saved submission.csv + oof_knn.npy/pred_knn.npy
